In [1]:
#!/usr/bin/env python3
"""
Fit the no-lapse DDM with sv and sz varying BY SAT CONDITION (speed vs.
accuracy), instead of one shared value across both conditions.

Rationale: sv shapes the slow-error tail, sz shapes the fast-error tail,
and the whole point of the speed/accuracy manipulation is to shift the
balance between fast and slow errors. Forcing one shared sv/sz to
explain both conditions at once may be exactly what pushes MAP toward
the degenerate near-zero compromise seen in the shared-parameter model.

Uses the same trimmed-data / hard-t0 setup as the other no-lapse cell.

OUTPUT FILE:
  - fits_ddm_nolapse_svsz_cond.csv -- one row per (participant, n_levels)
"""

import os
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

from datetime import datetime, timezone

import numpy as np
import pandas as pd
from cmdstanpy import CmdStanModel

# ╔═══════════════════════════════════════════════════════════╗
# ║  CONFIGURATION — edit these                              ║
# ╚═══════════════════════════════════════════════════════════╝
N_LEVELS      = 16
DATA_PATH     = "../../rr98.csv"
STAN_DDM      = "DDM_rr98_nolapse_svsz_by_cond.stan"
PARTICIPANTS  = ["jf", "kr", "nh"]
TRIM_LOW      = 0.01
TRIM_HIGH     = 0.99

DDM_OUT = "fits_ddm_nolapse_svsz_cond.csv"


def load_data():
    df = pd.read_csv(DATA_PATH)
    df = df[df["outlier"] == False].copy()
    df["correct"] = df["correct"].astype(int)
    df["sat_id"] = df["instruction"].map({"speed": 1, "accuracy": 2})
    df = df[df["strength"] != 16].copy()
    df["act_correct"] = ((df["strength"] > 16) == (df["response"] == "light")).astype(int)

    if N_LEVELS == 33:
        unique_strengths = sorted(df["strength"].unique())
        strength_to_level = {s: i+1 for i, s in enumerate(unique_strengths)}
        df["diff_level"] = df["strength"].map(strength_to_level)
        actual_levels = len(unique_strengths)
    else:
        def _qcut_levels(s):
            return pd.qcut(s, q=N_LEVELS, labels=False, duplicates="drop") + 1
        df["diff_level"] = df.groupby("id")["strength"].transform(_qcut_levels)
        actual_levels = df["diff_level"].nunique()

    df["cell"] = (df["sat_id"] - 1) * actual_levels + df["diff_level"]
    print(f"N_LEVELS requested: {N_LEVELS}, actual unique levels: {actual_levels}")
    print(f"Trials before trimming: {len(df)}")
    return df, actual_levels


def trim_extremes(df, low=TRIM_LOW, high=TRIM_HIGH):
    def _trim_group(g):
        lo, hi = g["rt"].quantile([low, high])
        return g[(g["rt"] >= lo) & (g["rt"] <= hi)]
    trimmed = df.groupby(["id", "cell"], group_keys=False).apply(_trim_group)
    print(f"Trials after trimming: {len(trimmed)} (dropped {len(df) - len(trimmed)})")
    return trimmed


def build_data(df, pid, n_levels):
    d = df[df["id"] == pid]
    d_correct = d[d["act_correct"] == 1]
    d_false = d[d["act_correct"] == 0]
    t0_hi = float(d["rt"].min())
    return {
        "N_LEVELS": n_levels,
        "N_correct": len(d_correct), "N_false": len(d_false),
        "rt_correct": d_correct["rt"].to_numpy(),
        "rt_false": d_false["rt"].to_numpy(),
        "cell_correct": d_correct["cell"].to_numpy(dtype=int),
        "cell_false": d_false["cell"].to_numpy(dtype=int),
        "t0_hi": t0_hi,
    }


def fit_ddm(model, data):
    nl = data["N_LEVELS"]
    inits = {
        "a": [0.8, 1.5], "v_base": [2.0]*nl,
        "sv": [0.5, 0.5], "sz": [0.1, 0.1],
        "t0": 0.2 * data["t0_hi"],
    }
    return model.optimize(data=data, inits=inits, algorithm="lbfgs",
                          iter=5000, show_console=True)


def aic_bic(mle, n_params):
    p = mle.optimized_params_pd
    ll_cols = [c for c in p.columns if c.startswith("log_lik")]
    total_ll = p[ll_cols].iloc[0].sum()
    n = len(ll_cols)
    return {"n_params": n_params, "n_trials": n, "log_lik": total_ll,
            "AIC": 2*n_params - 2*total_ll,
            "BIC": n_params*np.log(n) - 2*total_ll}


def save_fit_row(csv_path, pid, n_levels_requested, n_levels_actual, mle, ic, extra=None):
    raw = mle.optimized_params_pd.iloc[0]
    keep_cols = [c for c in raw.index if not c.startswith("log_lik")]
    row = raw[keep_cols].to_dict()
    row = {
        "pid": pid, "n_levels_requested": n_levels_requested,
        "n_levels_actual": n_levels_actual, "n_params": ic["n_params"],
        "n_trials": ic["n_trials"], "log_lik_total": ic["log_lik"],
        "AIC": ic["AIC"], "BIC": ic["BIC"],
        "fit_time_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        **(extra or {}), **row,
    }
    new_row = pd.DataFrame([row])
    if os.path.exists(csv_path):
        existing = pd.read_csv(csv_path)
        mask_same = (existing["pid"] == pid) & (existing["n_levels_actual"] == n_levels_actual)
        existing = existing[~mask_same]
        combined = pd.concat([existing, new_row], ignore_index=True, sort=False)
    else:
        combined = new_row
    combined = combined.sort_values(["n_levels_actual", "pid"]).reset_index(drop=True)
    combined.to_csv(csv_path, index=False)
    return combined


def main():
    df, actual_levels = load_data()
    df = trim_extremes(df)

    # a[2] + v_base[levels] + sv[2] + sz[2] + t0
    ddm_n_params = 2 + actual_levels + 2 + 2 + 1

    print(f"\nDDM (sv/sz by condition) params: {ddm_n_params} "
          f"({actual_levels} drift rates, sv/sz split by SAT)")

    print("\nCompiling model...")
    ddm_model = CmdStanModel(stan_file=STAN_DDM)

    for pid in PARTICIPANTS:
        print(f"\n{'='*60}\n  {pid}  (N_LEVELS={actual_levels}, sv/sz by condition)\n{'='*60}")

        data = build_data(df, pid, actual_levels)
        print(f"  N_correct={data['N_correct']}  N_false={data['N_false']}  "
              f"t0_hi={data['t0_hi']:.4f}")

        ddm_mle = fit_ddm(ddm_model, data)
        ic = aic_bic(ddm_mle, ddm_n_params)
        save_fit_row(DDM_OUT, pid, N_LEVELS, actual_levels, ddm_mle, ic,
                     extra={"t0_hi": data["t0_hi"]})
        row = ddm_mle.optimized_params_pd.iloc[0]
        print(f"  a=[{row['a[1]']:.3f}, {row['a[2]']:.3f}]  t0={row['t0']:.4f}")
        print(f"  sv=[speed={row['sv[1]']:.4f}, acc={row['sv[2]']:.4f}]")
        print(f"  sz=[speed={row['sz[1]']:.4f}, acc={row['sz[2]']:.4f}]")
        print(f"  LL={ic['log_lik']:.1f}  AIC={ic['AIC']:.1f}  BIC={ic['BIC']:.1f}")

    print(f"\nAll fits written/updated in:\n  {DDM_OUT}")


if __name__ == "__main__":
    main()


/var/folders/g5/d_krhx3n0xg4nf8x8frstwdw0000gq/T/ipykernel_84613/2615767332.py:69: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  trimmed = df.groupby(["id", "cell"], group_keys=False).apply(_trim_group)
15:23:09 - cmdstanpy - INFO - compiling stan file /Users/Anton/Documents/ForskningsprojektSommer2026/RR98fit/DDM/SvSzbyCond/DDM_rr98_nolapse_svsz_by_cond.stan to exe file /Users/Anton/Documents/ForskningsprojektSommer2026/RR98fit/DDM/SvSzbyCond/DDM_rr98_nolapse_svsz_by_cond


N_LEVELS requested: 16, actual unique levels: 16
Trials before trimming: 22584
Trials after trimming: 22060 (dropped 524)

DDM (sv/sz by condition) params: 23 (16 drift rates, sv/sz split by SAT)

Compiling model...


15:23:22 - cmdstanpy - INFO - compiled model executable: /Users/Anton/Documents/ForskningsprojektSommer2026/RR98fit/DDM/SvSzbyCond/DDM_rr98_nolapse_svsz_by_cond
15:23:22 - cmdstanpy - INFO - Chain [1] start processing



  jf  (N_LEVELS=16, sv/sz by condition)
  N_correct=6264  N_false=894  t0_hi=0.2080
Chain [1] method = optimize
Chain [1] optimize
Chain [1] algorithm = lbfgs (Default)
Chain [1] lbfgs
Chain [1] init_alpha = 0.001 (Default)
Chain [1] tol_obj = 1e-12 (Default)
Chain [1] tol_rel_obj = 10000 (Default)
Chain [1] tol_grad = 1e-08 (Default)
Chain [1] tol_rel_grad = 1e+07 (Default)
Chain [1] tol_param = 1e-08 (Default)
Chain [1] history_size = 5 (Default)
Chain [1] jacobian = false (Default)
Chain [1] iter = 5000
Chain [1] save_iterations = false (Default)
Chain [1] id = 1 (Default)
Chain [1] data
Chain [1] file = /var/folders/g5/d_krhx3n0xg4nf8x8frstwdw0000gq/T/tmpjy8i_l_f/y4xqinjy.json
Chain [1] init = /var/folders/g5/d_krhx3n0xg4nf8x8frstwdw0000gq/T/tmpjy8i_l_f/xbgdwze6.json
Chain [1] random
Chain [1] seed = 66482
Chain [1] output
Chain [1] file = /var/folders/g5/d_krhx3n0xg4nf8x8frstwdw0000gq/T/tmpjy8i_l_f/DDM_rr98_nolapse_svsz_by_condjmb8xvbk/DDM_rr98_nolapse_svsz_by_cond-20260722152322

15:24:03 - cmdstanpy - INFO - Chain [1] done processing


...

  a=[0.806, 2.162]  t0=0.2049
  sv=[speed=0.0057, acc=0.0016]
  sz=[speed=0.1650, acc=0.0269]
  LL=2540.6  AIC=-5035.1  BIC=-4877.0

  kr  (N_LEVELS=16, sv/sz by condition)
  N_correct=6089  N_false=934  t0_hi=0.2060
Chain [1] method = optimize
Chain [1] optimize
Chain [1] algorithm = lbfgs (Default)
Chain [1] lbfgs
Chain [1] init_alpha = 0.001 (Default)
Chain [1] tol_obj = 1e-12 (Default)
Chain [1] tol_rel_obj = 10000 (Default)
Chain [1] tol_grad = 1e-08 (Default)
Chain [1] tol_rel_grad = 1e+07 (Default)
Chain [1] tol_param = 1e-08 (Default)
Chain [1] history_size = 5 (Default)
Chain [1] jacobian = false (Default)
Chain [1] iter = 5000
Chain [1] save_iterations = false (Default)
Chain [1] id = 1 (Default)
Chain [1] data
Chain [1] file = /var/folders/g5/d_krhx3n0xg4nf8x8frstwdw0000gq/T/tmpjy8i_l_f/ygl1jgal.json
Chain [1] init = /var/folders/g5/d_krhx3n0xg4nf8x8frstwdw0000gq/T/tmpjy8i_l_f/x3qe3tav.json
Chain [1] random
Chain [1] seed = 16232
Chain [1] output
Chain [1] file = /var/folder

15:24:32 - cmdstanpy - INFO - Chain [1] done processing
15:24:33 - cmdstanpy - INFO - Chain [1] start processing


Chain [1] 327       2972.24   0.000212306      0.118595       0.924       0.924      367
Chain [1] Optimization terminated normally:
Chain [1] Convergence detected: relative gradient magnitude is below tolerance
Chain [1] 
  a=[0.770, 2.084]  t0=0.2029
  sv=[speed=0.0039, acc=0.0026]
  sz=[speed=0.2217, acc=0.0242]
  LL=2990.3  AIC=-5934.6  BIC=-5776.9

  nh  (N_LEVELS=16, sv/sz by condition)
  N_correct=7251  N_false=628  t0_hi=0.2260
Chain [1] method = optimize
Chain [1] optimize
Chain [1] algorithm = lbfgs (Default)
Chain [1] lbfgs
Chain [1] init_alpha = 0.001 (Default)
Chain [1] tol_obj = 1e-12 (Default)
Chain [1] tol_rel_obj = 10000 (Default)
Chain [1] tol_grad = 1e-08 (Default)
Chain [1] tol_rel_grad = 1e+07 (Default)
Chain [1] tol_param = 1e-08 (Default)
Chain [1] history_size = 5 (Default)
Chain [1] jacobian = false (Default)
Chain [1] iter = 5000
Chain [1] save_iterations = false (Default)
Chain [1] id = 1 (Default)
Chain [1] data
Chain [1] file = /var/folders/g5/d_krhx3n0xg4n

15:25:04 - cmdstanpy - INFO - Chain [1] done processing


Chain [1] 361       4804.77   0.000411519      0.118297           1           1      395
Chain [1] Optimization terminated normally:
Chain [1] Convergence detected: relative gradient magnitude is below tolerance
Chain [1] 
  a=[0.961, 1.773]  t0=0.2232
  sv=[speed=0.0026, acc=0.0023]
  sz=[speed=0.1164, acc=0.0519]
  LL=4820.5  AIC=-9594.9  BIC=-9434.6

All fits written/updated in:
  fits_ddm_nolapse_svsz_cond.csv


In [2]:
df = pd.read_csv("rr98.csv")
dude = "nk"
cond = "speed"
df[(df['id']==dude) & (df['instruction']==cond) & (df['correct']==True)]['rt'].hist(bins=100,density=True,alpha=0.5)
df[(df['id']==dude) & (df['instruction']==cond) & (df['correct']==False)]['rt'].hist(bins=100,density=True,alpha=0.5)
jf_speed = df[(df['id']==dude) & (df['instruction']==cond)]['rt']
print(jf_speed.quantile(0.05))
print((jf_speed < 0.15).sum())  # how many trials are faster than 150ms - implausible as genuine decisions

print(np.mean(df[(df['id']==dude) & (df['instruction']==cond) & (df['correct']==True)]['rt']))
print(np.mean(df[(df['id']==dude) & (df['instruction']==cond) & (df['correct']==False)]['rt']))

FileNotFoundError: [Errno 2] No such file or directory: 'rr98.csv'

In [4]:
#!/usr/bin/env python3
"""
Profile likelihood over a grid of FIXED sv values for the no-lapse DDM.

For each participant and each fixed sv in the grid, refits everything
else (a, v_base, sz, t0) via MAP, and records the total log-likelihood.
This answers directly: does the likelihood, with everything else allowed
to re-optimize, actually keep improving as sv -> 0 (genuine, persistent
pull toward the boundary), or is there an interior maximum somewhere
that the joint optimizer has been missing?

Run this on your machine (needs cmdstanpy + a compiled CmdStan + rr98.csv).

OUTPUT: profile_sv.csv, one row per (pid, sv_fixed).
"""

import os
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

from pathlib import Path

import numpy as np
import pandas as pd
from cmdstanpy import CmdStanModel

N_LEVELS = 16
STAN_DDM = "DDM_rr98_nolapse_profile_sv.stan"
PARTICIPANTS = ["jf", "kr", "nh"]
TRIM_LOW, TRIM_HIGH = 0.01, 0.99





DATA_PATH = "../../rr98.csv"
print(f"Using data file: {DATA_PATH}")

# The grid to profile over. Dense near 0 (where the action is), sparser
# further out. Add/remove points as you like -- denser = smoother curve,
# more compute.
SV_GRID = np.concatenate([
    [1e-4, 0.01, 0.02, 0.03, 0.05],
    np.arange(0.1, 1.01, 0.1),
])

OUT_PATH = "profile_sv.csv"


def load_data():
    df = pd.read_csv(DATA_PATH)
    required_cols = {"id", "outlier", "instruction", "strength", "response", "rt"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(
            f"Loaded {DATA_PATH}, but it's missing expected column(s): {missing}. "
            f"Columns found: {list(df.columns)}. This usually means DATA_PATH "
            f"resolved to the wrong file -- check find_rr98_csv() above."
        )
    df = df[df["outlier"] == False].copy()
    df["sat_id"] = df["instruction"].map({"speed": 1, "accuracy": 2})
    df = df[df["strength"] != 16].copy()
    df["act_correct"] = ((df["strength"] > 16) == (df["response"] == "light")).astype(int)

    def _qcut_levels(s):
        return pd.qcut(s, q=N_LEVELS, labels=False, duplicates="drop") + 1
    df["diff_level"] = df.groupby("id")["strength"].transform(_qcut_levels)
    actual_levels = df["diff_level"].nunique()
    df["cell"] = (df["sat_id"] - 1) * actual_levels + df["diff_level"]
    return df, actual_levels


def trim_extremes(df, low=TRIM_LOW, high=TRIM_HIGH):
    def _trim_group(g):
        lo, hi = g["rt"].quantile([low, high])
        return g[(g["rt"] >= lo) & (g["rt"] <= hi)]
    return df.groupby(["id", "cell"], group_keys=False).apply(_trim_group)


def build_data(df, pid, n_levels, sv_fixed):
    d = df[df["id"] == pid]
    d_correct = d[d["act_correct"] == 1]
    d_false = d[d["act_correct"] == 0]
    t0_hi = float(d["rt"].min())
    return {
        "N_LEVELS": n_levels,
        "N_correct": len(d_correct), "N_false": len(d_false),
        "rt_correct": d_correct["rt"].to_numpy(),
        "rt_false": d_false["rt"].to_numpy(),
        "cell_correct": d_correct["cell"].to_numpy(dtype=int),
        "cell_false": d_false["cell"].to_numpy(dtype=int),
        "t0_hi": t0_hi,
        "sv_fixed": float(sv_fixed),
    }


def fit_at_fixed_sv(model, data):
    nl = data["N_LEVELS"]
    inits = {"a": [0.8, 1.5], "v_base": [2.0]*nl, "sz": 0.1,
             "t0": 0.2 * data["t0_hi"]}
    return model.optimize(data=data, inits=inits, algorithm="lbfgs",
                          iter=5000, show_console=False)


def total_log_lik(mle):
    p = mle.optimized_params_pd
    ll_cols = [c for c in p.columns if c.startswith("log_lik")]
    return p[ll_cols].iloc[0].sum()


def main():
    df, actual_levels = load_data()
    df = trim_extremes(df)
    print(f"Using {actual_levels} difficulty levels, {len(SV_GRID)} sv values, "
          f"{len(PARTICIPANTS)} participants -> {len(SV_GRID)*len(PARTICIPANTS)} fits total.")

    print("Compiling model...")
    model = CmdStanModel(stan_file=STAN_DDM)

    rows = []
    for pid in PARTICIPANTS:
        print(f"\n=== {pid} ===")
        for sv_fixed in SV_GRID:
            data = build_data(df, pid, actual_levels, sv_fixed)
            try:
                mle = fit_at_fixed_sv(model, data)
                ll = total_log_lik(mle)
                row = mle.optimized_params_pd.iloc[0]
                print(f"  sv={sv_fixed:.4f}  log_lik={ll:.2f}  "
                      f"sz={row['sz']:.4f}  t0={row['t0']:.4f}")
                rows.append({"pid": pid, "sv_fixed": sv_fixed, "log_lik": ll,
                             "sz": row["sz"], "t0": row["t0"],
                             "a1": row["a[1]"], "a2": row["a[2]"]})
            except Exception as e:
                print(f"  sv={sv_fixed:.4f}  FAILED: {e}")
                rows.append({"pid": pid, "sv_fixed": sv_fixed, "log_lik": np.nan,
                             "sz": np.nan, "t0": np.nan, "a1": np.nan, "a2": np.nan})

    out = pd.DataFrame(rows)
    out.to_csv(OUT_PATH, index=False)
    print(f"\nSaved {OUT_PATH}")

    # Quick summary: where does the profile peak for each participant?
    for pid in PARTICIPANTS:
        sub = out[out["pid"] == pid].dropna()
        best = sub.loc[sub["log_lik"].idxmax()]
        print(f"{pid}: profile-likelihood peak at sv={best['sv_fixed']:.4f} "
              f"(log_lik={best['log_lik']:.2f})")


if __name__ == "__main__":
    main()


Using data file: ../../rr98.csv
Using 16 difficulty levels, 15 sv values, 3 participants -> 45 fits total.
Compiling model...

=== jf ===


/var/folders/g5/d_krhx3n0xg4nf8x8frstwdw0000gq/T/ipykernel_84613/346403415.py:76: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return df.groupby(["id", "cell"], group_keys=False).apply(_trim_group)
16:40:30 - cmdstanpy - INFO - Chain [1] start processing
16:40:41 - cmdstanpy - INFO - Chain [1] done processing
16:40:41 - cmdstanpy - INFO - Chain [1] start processing


  sv=0.0001  log_lik=2535.80  sz=0.0453  t0=0.2042


16:40:52 - cmdstanpy - INFO - Chain [1] done processing
16:40:52 - cmdstanpy - INFO - Chain [1] start processing


  sv=0.0100  log_lik=2535.34  sz=0.0457  t0=0.2042


...

  sv=0.8000  log_lik=2892.73  sz=0.0279  t0=0.2240


16:55:44 - cmdstanpy - INFO - Chain [1] done processing
16:55:44 - cmdstanpy - INFO - Chain [1] start processing


  sv=0.9000  log_lik=2492.93  sz=0.0268  t0=0.2241


16:55:58 - cmdstanpy - INFO - Chain [1] done processing


  sv=1.0000  log_lik=2082.22  sz=0.0259  t0=0.2242

Saved profile_sv.csv
jf: profile-likelihood peak at sv=0.0001 (log_lik=2535.80)
kr: profile-likelihood peak at sv=0.0001 (log_lik=2980.82)
nh: profile-likelihood peak at sv=0.0001 (log_lik=4819.49)


In [5]:
import pandas as pd

df = pd.read_csv("../../rr98.csv")   # adjust path as needed
df = df[df["outlier"] == False].copy()
df["correct"] = df["correct"].astype(bool)

print("=== Mean/median RT: correct vs. error, by participant ===")
print(df.groupby(["id", "correct"])["rt"].agg(["mean", "median", "count"]))

print("\n=== Mean/median RT: correct vs. error, by participant x instruction ===")
print(df.groupby(["id", "instruction", "correct"])["rt"].agg(["mean", "median", "count"]))

print("\n=== Same thing, but only for the hardest ~1/3 of trials (where errors concentrate) ===")
df["strength_dist"] = (df["strength"] - 16).abs()
hard = df[df["strength_dist"] <= df.groupby("id")["strength_dist"].transform(lambda s: s.quantile(0.33))]
print(hard.groupby(["id", "correct"])["rt"].agg(["mean", "median", "count"]))

=== Mean/median RT: correct vs. error, by participant ===
                mean  median  count
id correct                         
jf False    0.542559   0.395   2263
   True     0.524285   0.414   5472
kr False    0.581549   0.376   2216
   True     0.514998   0.378   5365
nh False    0.517855   0.416   2337
   True     0.462326   0.397   6195

=== Mean/median RT: correct vs. error, by participant x instruction ===
                            mean  median  count
id instruction correct                         
jf accuracy    False    0.794727  0.6760   1044
               True     0.717386  0.6230   2782
   speed       False    0.326591  0.3240   1219
               True     0.324578  0.3205   2690
kr accuracy    False    0.883381  0.6990   1046
               True     0.710465  0.5540   2739
   speed       False    0.311705  0.3060   1170
               True     0.311119  0.3030   2626
nh accuracy    False    0.678329  0.5500   1113
               True     0.566816  0.4910   3074
   sp

In [7]:
pip install hddm


[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [8]:
"""
Independent cross-check of the sv/sz findings using HDDM (Wiecki, Sofer &
Frank, 2013), which has sv/sz/st built in natively via the Navarro & Fuss
(2009) likelihood -- a different formula and implementation than our
Blurton et al. (2017) Stan code, and fit via full Bayesian MCMC (PyMC)
rather than MAP. If sv still concentrates near zero here, that's about as
strong an external confirmation as you can get without a third dataset.

INSTALL (can be finicky -- HDDM has fairly specific dependency pins):
    pip install hddm

If that fails, try:
    conda install -c conda-forge hddm
or check HDDM's GitHub for current install notes -- version conflicts
with newer numpy/scipy are the most common issue.
"""

import pandas as pd
import hddm

df = pd.read_csv("rr98.csv")
df = df[df["outlier"] == False].copy()

# HDDM expects: subj_idx, rt, response (1 = upper/correct boundary, 0 = lower)
df = df.rename(columns={"id": "subj_idx"})
df["response"] = (df["response"] == "dark").astype(int)  # pick one boundary as "1" consistently
df["condition"] = df["instruction"]

# Coarser binning to start with (5 bins), matching the rtdists reanalysis --
# deliberately coarser than your 16-level Stan fits, as a first check of
# whether sv recovers away from zero at this resolution, same as it did
# in the rtdists reanalysis.
df["strength_dist"] = df["strength"]
df["stim_bin"] = pd.cut(
    df["strength"], bins=[-0.5, 10.5, 13.5, 16.5, 19.5, 32.5],
    labels=["1", "2", "3", "4", "5"]
).astype(str)

# Full model: drift varies by stimulus bin, boundary/non-decision/sv/sz vary
# by speed-vs-accuracy condition -- same logical structure as your Stan model.
model = hddm.HDDM(
    df,
    depends_on={"v": "stim_bin", "a": "condition", "t": "condition"},
    include=("sv", "sz"),   # tells HDDM to estimate these rather than fix them
    p_outlier=0.0,          # no lapse mixture, matching your no-lapse variant
)

model.find_starting_values()
model.sample(3000, burn=500)   # bump these up for a real run; this is a quick check
model.print_stats()            # look at the sv, sz rows here

# Save the trace for later inspection / plotting
model.save("hddm_rr98_fit.db")

ModuleNotFoundError: No module named 'models'

In [10]:
conda create -n hddm python=3.8

ValueError: The python kernel does not appear to be a conda environment.  Please use ``%pip install`` instead.

In [15]:
#!/usr/bin/env python3
"""
Recreate the rtdists (Singmann et al.) reanalysis scheme for RR98 using
our own Stan/Blurton-density pipeline: same 5 strength bins, same
per-participant-x-instruction separate fits, raw response coding (not
correctness-recoded). If we reproduce their qualitative pattern (sv -> 0
in speed, sz speed > accuracy) with completely different code, that's
strong confirmation the result is about the data/model, not our pipeline.

OUTPUT: fits_reanalysis_style.csv, one row per (pid, instruction).
"""

import os
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from cmdstanpy import CmdStanModel

STAN_FILE = "DDM_rr98_reanalysis_style.stan"
PARTICIPANTS = ["jf", "kr", "nh"]
INSTRUCTIONS = ["speed", "accuracy"]
TRIM_LOW, TRIM_HIGH = 0.01, 0.99
APPLY_EXTRA_TRIM = False   # <-- set True to reproduce the earlier (trimmed) behavior.
                           #     False matches rtdists exactly: only the dataset's
                           #     own `outlier` column is excluded, no additional trim.

# Exact bin edges from the rtdists vignette (5 bins)
BIN_EDGES = [-0.5, 10.5, 13.5, 16.5, 19.5, 32.5]
N_LEVELS = len(BIN_EDGES) - 1

OUT_PATH = "fits_reanalysis_style_notrim.csv" if not APPLY_EXTRA_TRIM else "fits_reanalysis_style.csv"




def load_data():
    df = pd.read_csv("../../rr98.csv")
    df = df[df["outlier"] == False].copy()
    df["level"] = pd.cut(df["strength"], bins=BIN_EDGES, labels=False) + 1  # 1-indexed
    return df


def trim_extremes(df, low=TRIM_LOW, high=TRIM_HIGH):
    """Trim fastest/slowest tails within each (id, instruction, level, response)
    group -- as fine-grained as the earlier no-lapse trimming, just now also
    split by response identity since that's the unit of analysis here."""
    def _trim_group(g):
        lo, hi = g["rt"].quantile([low, high])
        return g[(g["rt"] >= lo) & (g["rt"] <= hi)]
    return df.groupby(["id", "instruction", "level"], group_keys=False).apply(_trim_group)


def build_data(df, pid, instruction):
    d = df[(df["id"] == pid) & (df["instruction"] == instruction)]
    d_dark = d[d["response"] == "dark"]
    d_light = d[d["response"] == "light"]
    t0_hi = float(d["rt"].min())
    return {
        "N_LEVELS": N_LEVELS,
        "N_dark": len(d_dark), "N_light": len(d_light),
        "rt_dark": d_dark["rt"].to_numpy(), "rt_light": d_light["rt"].to_numpy(),
        "level_dark": d_dark["level"].to_numpy(dtype=int),
        "level_light": d_light["level"].to_numpy(dtype=int),
        "t0_hi": t0_hi,
    }


def fit(model, data):
    inits = {"a": 1.5, "v": [0.0]*N_LEVELS, "z_rel": 0.5, "sv": 0.5, "sz": 0.1,
             "t0": 0.2 * data["t0_hi"]}
    return model.optimize(data=data, inits=inits, algorithm="lbfgs",
                          iter=5000, show_console=False)


def aic_bic(mle, n_params):
    p = mle.optimized_params_pd
    ll_cols = [c for c in p.columns if c.startswith("log_lik")]
    total_ll = p[ll_cols].iloc[0].sum()
    n = len(ll_cols)
    return {"n_params": n_params, "n_trials": n, "log_lik": total_ll,
            "AIC": 2*n_params - 2*total_ll, "BIC": n_params*np.log(n) - 2*total_ll}


def save_row(csv_path, pid, instruction, mle, ic, t0_hi):
    raw = mle.optimized_params_pd.iloc[0]
    keep_cols = [c for c in raw.index if not c.startswith("log_lik")]
    row = raw[keep_cols].to_dict()
    row = {"pid": pid, "instruction": instruction, "n_params": ic["n_params"],
           "n_trials": ic["n_trials"], "log_lik_total": ic["log_lik"],
           "AIC": ic["AIC"], "BIC": ic["BIC"], "t0_hi": t0_hi,
           "fit_time_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
           **row}
    new_row = pd.DataFrame([row])
    if os.path.exists(csv_path):
        existing = pd.read_csv(csv_path)
        mask = (existing["pid"] == pid) & (existing["instruction"] == instruction)
        existing = existing[~mask]
        combined = pd.concat([existing, new_row], ignore_index=True, sort=False)
    else:
        combined = new_row
    combined = combined.sort_values(["pid", "instruction"]).reset_index(drop=True)
    combined.to_csv(csv_path, index=False)


def main():
    df = load_data()
    if APPLY_EXTRA_TRIM:
        df = trim_extremes(df)
        print("Applied extra 1%/99% per-cell trim (on top of the dataset's own outlier flag).")
    else:
        print("No extra trim -- using only the dataset's own `outlier` flag, matching rtdists.")

    n_params = 1 + N_LEVELS + 1 + 1 + 1 + 1  # a, v[5], z_rel, sv, sz, t0
    print(f"Params per fit: {n_params} ({N_LEVELS} drift rates + a, z_rel, sv, sz, t0)")

    print("Compiling model...")
    model = CmdStanModel(stan_file=STAN_FILE)

    for pid in PARTICIPANTS:
        for instruction in INSTRUCTIONS:
            print(f"\n=== {pid} / {instruction} ===")
            data = build_data(df, pid, instruction)
            print(f"  N_dark={data['N_dark']}  N_light={data['N_light']}  "
                  f"t0_hi={data['t0_hi']:.4f}")
            mle = fit(model, data)
            ic = aic_bic(mle, n_params)
            save_row(OUT_PATH, pid, instruction, mle, ic, data["t0_hi"])
            row = mle.optimized_params_pd.iloc[0]
            print(f"  a={row['a']:.3f}  z_rel={row['z_rel']:.3f}  t0={row['t0']:.4f}  "
                  f"sv={row['sv']:.4f}  sz={row['sz']:.4f}")
            print(f"  LL={ic['log_lik']:.1f}  AIC={ic['AIC']:.1f}  BIC={ic['BIC']:.1f}")

    print(f"\nSaved: {OUT_PATH}")


if __name__ == "__main__":
    main()

12:47:01 - cmdstanpy - INFO - Chain [1] start processing


No extra trim -- using only the dataset's own `outlier` flag, matching rtdists.
Params per fit: 10 (5 drift rates + a, z_rel, sv, sz, t0)
Compiling model...

=== jf / speed ===
  N_dark=1959  N_light=1950  t0_hi=0.2000


12:47:07 - cmdstanpy - INFO - Chain [1] done processing
12:47:07 - cmdstanpy - INFO - Chain [1] start processing


  a=0.877  z_rel=0.468  t0=0.1955  sv=0.0040  sz=0.2506
  LL=3200.5  AIC=-6380.9  BIC=-6318.2

=== jf / accuracy ===
  N_dark=1823  N_light=2003  t0_hi=0.2340


12:47:12 - cmdstanpy - INFO - Chain [1] done processing
12:47:12 - cmdstanpy - INFO - Chain [1] start processing


...

  a=1.883  z_rel=0.546  t0=0.2135  sv=0.0002  sz=0.0392
  LL=-1326.4  AIC=2672.7  BIC=2735.1

=== nh / speed ===
  N_dark=2138  N_light=2207  t0_hi=0.2010


12:47:29 - cmdstanpy - INFO - Chain [1] done processing
12:47:29 - cmdstanpy - INFO - Chain [1] start processing


  a=1.129  z_rel=0.524  t0=0.1958  sv=0.0024  sz=0.1689
  LL=3318.5  AIC=-6616.9  BIC=-6553.2

=== nh / accuracy ===
  N_dark=2041  N_light=2146  t0_hi=0.2430


12:47:37 - cmdstanpy - INFO - Chain [1] done processing


  a=1.630  z_rel=0.517  t0=0.2235  sv=0.0013  sz=0.1473
  LL=185.0  AIC=-350.0  BIC=-286.6

Saved: fits_reanalysis_style_notrim.csv


In [17]:
#!/usr/bin/env python3
"""
Recreate the rtdists (Singmann et al.) reanalysis scheme for RR98 using
our own Stan/Blurton-density pipeline: same 5 strength bins, same
per-participant-x-instruction separate fits, raw response coding (not
correctness-recoded). If we reproduce their qualitative pattern (sv -> 0
in speed, sz speed > accuracy) with completely different code, that's
strong confirmation the result is about the data/model, not our pipeline.

OUTPUT: fits_reanalysis_style.csv, one row per (pid, instruction).
"""

import os
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from cmdstanpy import CmdStanModel

STAN_FILE = "DDM_rr98_reanalysis_style.stan"
PARTICIPANTS = ["jf", "kr", "nh"]
INSTRUCTIONS = ["speed", "accuracy"]
TRIM_LOW, TRIM_HIGH = 0.01, 0.99
APPLY_EXTRA_TRIM = False   # <-- set True to reproduce the earlier (trimmed) behavior.
                           #     False matches rtdists exactly: only the dataset's
                           #     own `outlier` column is excluded, no additional trim.

# Exact bin edges from the rtdists vignette (5 bins)
BIN_EDGES = [-0.5, 10.5, 13.5, 16.5, 19.5, 32.5]
N_LEVELS = len(BIN_EDGES) - 1

OUT_PATH = "fits_reanalysis_style_notrim.csv" if not APPLY_EXTRA_TRIM else "fits_reanalysis_style.csv"



def load_data():
    df = pd.read_csv("../../rr98.csv")
    df = df[df["outlier"] == False].copy()
    df["level"] = pd.cut(df["strength"], bins=BIN_EDGES, labels=False) + 1  # 1-indexed
    return df


def trim_extremes(df, low=TRIM_LOW, high=TRIM_HIGH):
    """Trim fastest/slowest tails within each (id, instruction, level, response)
    group -- as fine-grained as the earlier no-lapse trimming, just now also
    split by response identity since that's the unit of analysis here."""
    def _trim_group(g):
        lo, hi = g["rt"].quantile([low, high])
        return g[(g["rt"] >= lo) & (g["rt"] <= hi)]
    return df.groupby(["id", "instruction", "level"], group_keys=False).apply(_trim_group)


def build_data(df, pid, instruction):
    d = df[(df["id"] == pid) & (df["instruction"] == instruction)]
    d_dark = d[d["response"] == "dark"]
    d_light = d[d["response"] == "light"]
    t0_hi = float(d["rt"].min())
    return {
        "N_LEVELS": N_LEVELS,
        "N_dark": len(d_dark), "N_light": len(d_light),
        "rt_dark": d_dark["rt"].to_numpy(), "rt_light": d_light["rt"].to_numpy(),
        "level_dark": d_dark["level"].to_numpy(dtype=int),
        "level_light": d_light["level"].to_numpy(dtype=int),
        "t0_hi": t0_hi,
    }


def fit(model, data, n_restarts=8, seed=None):
    """Multiple randomized-start optimization runs, keeping the best.
    Matches rtdists' strategy of trying several random starting values
    rather than trusting a single fixed init -- if there's a real but
    shallow interior peak in sv, a single init can easily miss it."""
    rng = np.random.default_rng(seed)
    best_mle = None
    best_ll = -np.inf
    for i in range(n_restarts):
        inits = {
            "a": float(rng.uniform(0.5, 3.0)),
            "v": list(rng.normal(0, 1.5, size=N_LEVELS)),
            "z_rel": float(rng.uniform(0.35, 0.65)),
            "sv": float(rng.uniform(0.0, 1.5)),      # wide range, including large values
            "sz": float(rng.uniform(0.0, 0.5)),
            "t0": float(rng.uniform(0.05, 0.9 * data["t0_hi"])),
        }
        try:
            mle = model.optimize(data=data, inits=inits, algorithm="lbfgs",
                                 iter=5000, show_console=False)
            p = mle.optimized_params_pd
            ll_cols = [c for c in p.columns if c.startswith("log_lik")]
            ll = p[ll_cols].iloc[0].sum()
            print(f"    restart {i}: sv_init={inits['sv']:.3f} -> "
                  f"sv_fit={p['sv'].iloc[0]:.4f}  LL={ll:.2f}")
            if ll > best_ll:
                best_ll = ll
                best_mle = mle
        except Exception as e:
            print(f"    restart {i}: FAILED ({e})")
    return best_mle


def aic_bic(mle, n_params):
    p = mle.optimized_params_pd
    ll_cols = [c for c in p.columns if c.startswith("log_lik")]
    total_ll = p[ll_cols].iloc[0].sum()
    n = len(ll_cols)
    return {"n_params": n_params, "n_trials": n, "log_lik": total_ll,
            "AIC": 2*n_params - 2*total_ll, "BIC": n_params*np.log(n) - 2*total_ll}


def save_row(csv_path, pid, instruction, mle, ic, t0_hi):
    raw = mle.optimized_params_pd.iloc[0]
    keep_cols = [c for c in raw.index if not c.startswith("log_lik")]
    row = raw[keep_cols].to_dict()
    row = {"pid": pid, "instruction": instruction, "n_params": ic["n_params"],
           "n_trials": ic["n_trials"], "log_lik_total": ic["log_lik"],
           "AIC": ic["AIC"], "BIC": ic["BIC"], "t0_hi": t0_hi,
           "fit_time_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
           **row}
    new_row = pd.DataFrame([row])
    if os.path.exists(csv_path):
        existing = pd.read_csv(csv_path)
        mask = (existing["pid"] == pid) & (existing["instruction"] == instruction)
        existing = existing[~mask]
        combined = pd.concat([existing, new_row], ignore_index=True, sort=False)
    else:
        combined = new_row
    combined = combined.sort_values(["pid", "instruction"]).reset_index(drop=True)
    combined.to_csv(csv_path, index=False)


def main():
    df = load_data()
    if APPLY_EXTRA_TRIM:
        df = trim_extremes(df)
        print("Applied extra 1%/99% per-cell trim (on top of the dataset's own outlier flag).")
    else:
        print("No extra trim -- using only the dataset's own `outlier` flag, matching rtdists.")

    n_params = 1 + N_LEVELS + 1 + 1 + 1 + 1  # a, v[5], z_rel, sv, sz, t0
    print(f"Params per fit: {n_params} ({N_LEVELS} drift rates + a, z_rel, sv, sz, t0)")

    print("Compiling model...")
    model = CmdStanModel(stan_file=STAN_FILE)

    for pid in PARTICIPANTS:
        for instruction in INSTRUCTIONS:
            print(f"\n=== {pid} / {instruction} ===")
            data = build_data(df, pid, instruction)
            print(f"  N_dark={data['N_dark']}  N_light={data['N_light']}  "
                  f"t0_hi={data['t0_hi']:.4f}")
            mle = fit(model, data)
            ic = aic_bic(mle, n_params)
            save_row(OUT_PATH, pid, instruction, mle, ic, data["t0_hi"])
            row = mle.optimized_params_pd.iloc[0]
            print(f"  a={row['a']:.3f}  z_rel={row['z_rel']:.3f}  t0={row['t0']:.4f}  "
                  f"sv={row['sv']:.4f}  sz={row['sz']:.4f}")
            print(f"  LL={ic['log_lik']:.1f}  AIC={ic['AIC']:.1f}  BIC={ic['BIC']:.1f}")

    print(f"\nSaved: {OUT_PATH}")


if __name__ == "__main__":
    main()

12:52:30 - cmdstanpy - INFO - Chain [1] start processing


No extra trim -- using only the dataset's own `outlier` flag, matching rtdists.
Params per fit: 10 (5 drift rates + a, z_rel, sv, sz, t0)
Compiling model...

=== jf / speed ===
  N_dark=1959  N_light=1950  t0_hi=0.2000


12:52:35 - cmdstanpy - INFO - Chain [1] done processing
12:52:35 - cmdstanpy - INFO - Chain [1] start processing


    restart 0: sv_init=0.652 -> sv_fit=0.0043  LL=3200.46


12:52:39 - cmdstanpy - INFO - Chain [1] done processing
12:52:39 - cmdstanpy - INFO - Chain [1] start processing


...

    restart 5: sv_init=1.167 -> sv_fit=0.0012  LL=184.98


12:58:21 - cmdstanpy - INFO - Chain [1] done processing
12:58:21 - cmdstanpy - INFO - Chain [1] start processing


    restart 6: sv_init=0.553 -> sv_fit=0.0014  LL=184.98


12:58:27 - cmdstanpy - INFO - Chain [1] done processing


    restart 7: sv_init=0.950 -> sv_fit=0.0012  LL=184.98
  a=1.630  z_rel=0.517  t0=0.2235  sv=0.0012  sz=0.1473
  LL=185.0  AIC=-350.0  BIC=-286.6

Saved: fits_reanalysis_style_notrim.csv


In [19]:
#!/usr/bin/env python3
"""
Same reanalysis-style scheme (5 rtdists bins, fit separately per
participant x instruction, raw response coding, no extra trim) but using
Stan's own NATIVE wiener_lpdf() density instead of our hand-written
Blurton closed-form. If sv comes out meaningfully different here, the
density implementation itself is the answer; if it still collapses near
zero, that's about as strong a confirmation as this investigation can
get that the result is real and not an artifact of our code.

REQUIRES CmdStan >= ~2.36 (built and confirmed working against 2.39.0).
If you're on an older CmdStan, upgrade first:
    import cmdstanpy
    cmdstanpy.install_cmdstan(version="2.39.0")
    cmdstanpy.set_cmdstan_path("<path>/.cmdstan/cmdstan-2.39.0")

OUTPUT: fits_reanalysis_native.csv, one row per (pid, instruction).
"""

import os
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from cmdstanpy import CmdStanModel

STAN_FILE = "DDM_rr98_reanalysis_style_native.stan"
PARTICIPANTS = ["jf", "kr", "nh"]
INSTRUCTIONS = ["speed", "accuracy"]

BIN_EDGES = [-0.5, 10.5, 13.5, 16.5, 19.5, 32.5]
N_LEVELS = len(BIN_EDGES) - 1

OUT_PATH = "fits_reanalysis_native.csv"




def load_data():
    df = pd.read_csv("../../rr98.csv")
    df = df[df["outlier"] == False].copy()
    df["level"] = pd.cut(df["strength"], bins=BIN_EDGES, labels=False) + 1
    return df


def build_data(df, pid, instruction):
    d = df[(df["id"] == pid) & (df["instruction"] == instruction)]
    d_dark = d[d["response"] == "dark"]
    d_light = d[d["response"] == "light"]
    t0_hi = float(d["rt"].min())
    return {
        "N_LEVELS": N_LEVELS,
        "N_dark": len(d_dark), "N_light": len(d_light),
        "rt_dark": d_dark["rt"].to_numpy(), "rt_light": d_light["rt"].to_numpy(),
        "level_dark": d_dark["level"].to_numpy(dtype=int),
        "level_light": d_light["level"].to_numpy(dtype=int),
        "t0_hi": t0_hi,
    }


def fit(model, data, n_restarts=8, seed=None):
    """Same multi-restart strategy as the custom-density version, for a
    fair comparison -- rules out convergence differences as a confound."""
    rng = np.random.default_rng(seed)
    best_mle, best_ll = None, -np.inf
    for i in range(n_restarts):
        inits = {
            "a": float(rng.uniform(0.5, 3.0)),
            "v": list(rng.normal(0, 1.5, size=N_LEVELS)),
            "z_rel": float(rng.uniform(0.35, 0.65)),
            "sv": float(rng.uniform(0.0, 1.5)),
            "sz": float(rng.uniform(0.0, 0.5)),
            "t0": float(rng.uniform(0.05, 0.9 * data["t0_hi"])),
        }
        try:
            mle = model.optimize(data=data, inits=inits, algorithm="lbfgs",
                                 iter=5000, show_console=False)
            p = mle.optimized_params_pd
            ll_cols = [c for c in p.columns if c.startswith("log_lik")]
            ll = p[ll_cols].iloc[0].sum()
            print(f"    restart {i}: sv_init={inits['sv']:.3f} -> "
                  f"sv_fit={p['sv'].iloc[0]:.4f}  LL={ll:.2f}")
            if ll > best_ll:
                best_ll, best_mle = ll, mle
        except Exception as e:
            print(f"    restart {i}: FAILED ({e})")
    return best_mle


def aic_bic(mle, n_params):
    p = mle.optimized_params_pd
    ll_cols = [c for c in p.columns if c.startswith("log_lik")]
    total_ll = p[ll_cols].iloc[0].sum()
    n = len(ll_cols)
    return {"n_params": n_params, "n_trials": n, "log_lik": total_ll,
            "AIC": 2*n_params - 2*total_ll, "BIC": n_params*np.log(n) - 2*total_ll}


def save_row(csv_path, pid, instruction, mle, ic, t0_hi):
    raw = mle.optimized_params_pd.iloc[0]
    keep_cols = [c for c in raw.index if not c.startswith("log_lik")]
    row = raw[keep_cols].to_dict()
    row = {"pid": pid, "instruction": instruction, "n_params": ic["n_params"],
           "n_trials": ic["n_trials"], "log_lik_total": ic["log_lik"],
           "AIC": ic["AIC"], "BIC": ic["BIC"], "t0_hi": t0_hi,
           "fit_time_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
           **row}
    new_row = pd.DataFrame([row])
    if os.path.exists(csv_path):
        existing = pd.read_csv(csv_path)
        mask = (existing["pid"] == pid) & (existing["instruction"] == instruction)
        existing = existing[~mask]
        combined = pd.concat([existing, new_row], ignore_index=True, sort=False)
    else:
        combined = new_row
    combined = combined.sort_values(["pid", "instruction"]).reset_index(drop=True)
    combined.to_csv(csv_path, index=False)


def main():
    df = load_data()
    n_params = 1 + N_LEVELS + 1 + 1 + 1 + 1
    print(f"Params per fit: {n_params}")
    print("Compiling model (native wiener_lpdf)...")
    model = CmdStanModel(stan_file=STAN_FILE)

    for pid in PARTICIPANTS:
        for instruction in INSTRUCTIONS:
            print(f"\n=== {pid} / {instruction} ===")
            data = build_data(df, pid, instruction)
            print(f"  N_dark={data['N_dark']}  N_light={data['N_light']}  "
                  f"t0_hi={data['t0_hi']:.4f}")
            mle = fit(model, data)
            ic = aic_bic(mle, n_params)
            save_row(OUT_PATH, pid, instruction, mle, ic, data["t0_hi"])
            row = mle.optimized_params_pd.iloc[0]
            print(f"  a={row['a']:.3f}  z_rel={row['z_rel']:.3f}  t0={row['t0']:.4f}  "
                  f"sv={row['sv']:.4f}  sz={row['sz']:.4f}")
            print(f"  LL={ic['log_lik']:.1f}  AIC={ic['AIC']:.1f}  BIC={ic['BIC']:.1f}")

    print(f"\nSaved: {OUT_PATH}")


if __name__ == "__main__":
    main()

13:16:30 - cmdstanpy - INFO - compiling stan file /Users/Anton/Documents/ForskningsprojektSommer2026/RR98fit/DDM/SvSzbyCond/DDM_rr98_reanalysis_style_native.stan to exe file /Users/Anton/Documents/ForskningsprojektSommer2026/RR98fit/DDM/SvSzbyCond/DDM_rr98_reanalysis_style_native


Params per fit: 10
Compiling model (native wiener_lpdf)...


13:16:39 - cmdstanpy - INFO - compiled model executable: /Users/Anton/Documents/ForskningsprojektSommer2026/RR98fit/DDM/SvSzbyCond/DDM_rr98_reanalysis_style_native
13:16:39 - cmdstanpy - INFO - Chain [1] start processing



=== jf / speed ===
  N_dark=1959  N_light=1950  t0_hi=0.2000


13:16:48 - cmdstanpy - INFO - Chain [1] done processing
13:16:48 - cmdstanpy - INFO - Chain [1] start processing


...

    restart 5: sv_init=1.240 -> sv_fit=0.9666  LL=245.00


13:30:44 - cmdstanpy - INFO - Chain [1] done processing
13:30:44 - cmdstanpy - INFO - Chain [1] start processing


    restart 6: sv_init=0.049 -> sv_fit=0.9666  LL=245.00


13:30:53 - cmdstanpy - INFO - Chain [1] done processing


    restart 7: sv_init=0.513 -> sv_fit=0.9665  LL=245.00
  a=1.827  z_rel=0.481  t0=0.2265  sv=0.9667  sz=0.2851
  LL=245.0  AIC=-470.0  BIC=-406.6

Saved: fits_reanalysis_native.csv
